# mKA Cavity MD Tutorial

This notebook walks through building cavity molecular-dynamics simulations of the **modified Kob–Andersen (mKA) dimer** model step by step.  
Each section adds one layer of physics so you can see exactly which OpenMM object does what.

| Section | System | Integrator |
|---------|--------|------------|
| 1 | Single A–A dimer + photon | NVE (Verlet) |
| 2 | Single A–A dimer + photon | NVT Bussi–Parrinello |
| 3 | Two dimers (A–A + B–B) + photon, LJ + Coulomb | NVT Bussi–Parrinello |
| 4 | N-many dimers (use the library builder) | NVT Bussi–Parrinello |
| 5 | Compute dipole-moment spectrum manually via FFT | — |

**Kernel**: activate the `test` pixi environment before launching Jupyter:  
```bash
pixi run -e test jupyter lab
```

---
### Unit conventions

The mKA force-field parameters are defined in **atomic units** (Hartree / Bohr / electron mass).  
OpenMM stores everything internally in **kJ/mol, nm, ps** — the conversions are done below.  
The `Units` helper class from `openmm.cavitymd.constants` contains all the factors you need.

## Section 0 — Imports and shared parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openmm
from openmm import unit

# Unit-conversion helpers
from openmm.cavitymd.constants import Units

# mKA force-field constants and builder functions
from openmm.cavitymd.forcefields.mka import (
    # Masses (amu)
    MASS_A, MASS_B, PHOTON_MASS_AMU,
    # Harmonic bond parameters (atomic units)
    K_AA_AU, R0_AA_AU, K_BB_AU, R0_BB_AU,
    # Lennard-Jones parameters (atomic units)
    EPS_AA_AU, SIG_AA_AU,
    EPS_AB_AU, SIG_AB_AU,
    EPS_BB_AU, SIG_BB_AU,
    # Charge magnitude (elementary charge)
    CHARGE_MAG,
    # Cavity
    OMEGA_C_CM1,
    # N-body builder helpers
    box_au_for_num_molecules, build_mka_system, add_cavity_particle,
)

# Bussi thermostat setup helper (thin wrapper around openmm.BussiThermostat)
from openmm.cavitymd.thermostats import DualThermostat

print("OpenMM version:", openmm.__version__)
for i in range(openmm.Platform.getNumPlatforms()):
    print(" platform:", openmm.Platform.getPlatform(i).getName())

In [ ]:
# ============================================================
# Tunable parameters — change these to explore the physics
# ============================================================
lambda_coupling = 0.01    # dimensionless coupling λ (a.u.) — try 0.0, 0.01, 0.03
T_K             = 100.0   # bath temperature (K)
dt_fs           = 1.0     # timestep (fs)
N_steps         = 20_000  # number of steps (~20 ps at 1 fs/step)
seed            = 42

# Derived / converted constants used throughout
dt_ps           = dt_fs * 1e-3                   # ps
omegac_au       = Units.cm1_to_au(OMEGA_C_CM1)   # cavity freq in Hartree

# Conversion shortcuts for bonds and LJ
B2NM = Units.BOHR_TO_NM
H2K  = Units.HARTREE_TO_KJMOL

# Platform: prefer CUDA (mixed precision), fall back to CPU
try:
    platform = openmm.Platform.getPlatformByName("CUDA")
    platform.setPropertyDefaultValue("Precision", "mixed")
    print("Using CUDA platform (mixed precision)")
except Exception:
    platform = openmm.Platform.getPlatformByName("CPU")
    print("Using CPU platform")

print(f"λ = {lambda_coupling},  T = {T_K} K,  dt = {dt_fs} fs,  ω_c = {OMEGA_C_CM1} cm⁻¹")
print(f"ω_c in a.u. = {omegac_au:.6e} Hartree")

---
## Section 1 — NVE single A–A dimer

The simplest possible cavity-MD system: **one A–A dimer** (two charged atoms) plus **one photon particle**.  
No inter-molecular forces yet — just the intramolecular harmonic bond and the cavity interaction.

### Forces
| OpenMM object | Role |
|---------------|------|
| `HarmonicBondForce` | A–A bond: `K_AA`, `r0_AA` |
| `CavityForce` | harmonic photon well + bilinear coupling + dipole self-energy |
| `CavityParticleDisplacer` | finite-q shift at initialisation |

No `NonbondedForce` yet — that comes in Section 3.

### Finite-q displacement
When the cavity coupling is turned on the photon's equilibrium position shifts to  
$$q_\text{eq} = -\frac{\lambda}{m_\text{ph}\,\omega_c}\,d_{xy}
      = -\frac{\epsilon}{K}\,d_{xy}$$  
where $d_{xy}$ is the molecular dipole projected onto the $(x,y)$ plane.  
Starting from $q=0$ would inject a large amount of energy into the photon mode;  
`CavityParticleDisplacer.displaceToEquilibrium()` moves the photon to $q_\text{eq}$ before the first step.

In [ ]:
# ── Build the OpenMM System by hand ──────────────────────────────────────
system_nve = openmm.System()

# Particle 0: A-atom (negative charge, −0.3e)
# Particle 1: A-atom (positive charge, +0.3e)
# Particle 2: photon
system_nve.addParticle(MASS_A)           # idx 0
system_nve.addParticle(MASS_A)           # idx 1
system_nve.addParticle(PHOTON_MASS_AMU)  # idx 2 — photon

# ── Harmonic bond (convert from a.u. to OpenMM units) ────────────────────
k_aa_omm  = K_AA_AU  * H2K / B2NM**2   # kJ/mol/nm²
r0_aa_omm = R0_AA_AU * B2NM             # nm

bond_force = openmm.HarmonicBondForce()
bond_force.addBond(0, 1, r0_aa_omm, k_aa_omm)
system_nve.addForce(bond_force)

# ── CavityForce ──────────────────────────────────────────────────────────
# CavityForce(cavity_particle_index, omegac_au, lambda, photon_mass_amu)
cavity_force_nve = openmm.CavityForce(2, omegac_au, lambda_coupling, PHOTON_MASS_AMU)
system_nve.addForce(cavity_force_nve)

# ── CavityParticleDisplacer (finite-q shift) ──────────────────────────────
displacer_nve = openmm.CavityParticleDisplacer(2, omegac_au, PHOTON_MASS_AMU)
# We call displaceToEquilibrium() manually; disable the automatic trigger.
displacer_nve.setSwitchOnStep(2**31 - 1)
system_nve.addForce(displacer_nve)

print(f"System has {system_nve.getNumParticles()} particles, {system_nve.getNumForces()} forces")

In [ ]:
# ── Initial positions ─────────────────────────────────────────────────────
# Place the dimer along the x-axis, centred at the origin.
# Atom 0 at -r0/2, atom 1 at +r0/2, photon starts at origin (displaced below).
half_r0 = r0_aa_omm / 2.0
positions_nve = [
    openmm.Vec3(-half_r0, 0, 0) * unit.nanometer,  # A−
    openmm.Vec3(+half_r0, 0, 0) * unit.nanometer,  # A+
    openmm.Vec3(      0,  0, 0) * unit.nanometer,  # photon (will be displaced)
]

# ── Integrator + Context ──────────────────────────────────────────────────
integrator_nve = openmm.VerletIntegrator(dt_ps)  # pure NVE
context_nve = openmm.Context(system_nve, integrator_nve, platform)
context_nve.setPositions(positions_nve)

# Thermal velocities (Maxwell–Boltzmann at T_K)
context_nve.setVelocitiesToTemperature(T_K, seed)

# Finite-q displacement: move photon to q_eq = -(lambda/(m_ph*omega_c)) d_xy
displacer_nve.displaceToEquilibrium(context_nve, lambda_coupling)

state = context_nve.getState(getPositions=True)
q_photon = state.getPositions(asNumpy=True)[2]   # nm
print(f"Photon position after finite-q shift: ({q_photon[0]:.4f}, {q_photon[1]:.4f}, {q_photon[2]:.4f}) nm")

In [ ]:
# ── Run NVE and collect the molecular dipole d(t) ─────────────────────────
charges_single = np.array([-CHARGE_MAG, +CHARGE_MAG])  # elementary charge

dipoles_nve = []
for step in range(N_steps):
    integrator_nve.step(1)
    state = context_nve.getState(getPositions=True)
    pos = state.getPositions(asNumpy=True)          # (3, 3) in nm
    # dipole = Σ q_i * r_i  [units: e·nm]
    d = charges_single[0] * pos[0] + charges_single[1] * pos[1]
    dipoles_nve.append(d)

dipoles_nve = np.array(dipoles_nve)   # shape (N_steps, 3)
t_ps = np.arange(N_steps) * dt_ps

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_ps, dipoles_nve[:, 0], lw=0.6, label='d_x')
ax.set_xlabel('time (ps)')
ax.set_ylabel('dipole (e·nm)')
ax.set_title('NVE single A–A dimer — x-component of dipole')
ax.legend()
plt.tight_layout()
plt.show()

**What to observe:**  
The dipole oscillates at (approximately) the A–A stretch frequency ω_c = 1560 cm⁻¹.  
Because there is no thermostat the total energy is conserved; the oscillation amplitude does not decay.

---
## Section 2 — NVT single A–A dimer (Bussi–Parrinello thermostat)

We add a **Bussi–Parrinello CSVR thermostat** to the molecular atoms only.  
The photon is *intentionally excluded*: it thermalises through its mechanical coupling to the molecular mode,
not by direct bath contact.  

> `openmm.BussiThermostat(T_K, tau_ps)` is a native OpenMM `Force` that rescales
> particle velocities stochastically at every step so the kinetic energy is sampled from the
> correct canonical distribution (Bussi et al., J. Chem. Phys. **126**, 014101, 2007).

In [ ]:
# ── Re-build the System (same as NVE but add BussiThermostat) ─────────────
system_nvt1 = openmm.System()
system_nvt1.addParticle(MASS_A)
system_nvt1.addParticle(MASS_A)
system_nvt1.addParticle(PHOTON_MASS_AMU)

bond_force_2 = openmm.HarmonicBondForce()
bond_force_2.addBond(0, 1, r0_aa_omm, k_aa_omm)
system_nvt1.addForce(bond_force_2)

cavity_force_nvt1 = openmm.CavityForce(2, omegac_au, lambda_coupling, PHOTON_MASS_AMU)
system_nvt1.addForce(cavity_force_nvt1)

displacer_nvt1 = openmm.CavityParticleDisplacer(2, omegac_au, PHOTON_MASS_AMU)
displacer_nvt1.setSwitchOnStep(2**31 - 1)
system_nvt1.addForce(displacer_nvt1)

# ── Bussi thermostat — ONLY on molecular atoms [0, 1] ────────────────────
# The helper adds the BussiThermostat Force and registers only the
# listed particle indices with it.
DualThermostat.setup_bussi_for_system(
    system_nvt1,
    molecular_indices=[0, 1],
    temperature_K=T_K,
    tau_ps=1.0,
    random_number_seed=seed,
)

print(f"System forces: {[type(system_nvt1.getForce(i)).__name__ for i in range(system_nvt1.getNumForces())]}")

In [ ]:
# ── Context + initial conditions ──────────────────────────────────────────
integrator_nvt1 = openmm.VerletIntegrator(dt_ps)
context_nvt1 = openmm.Context(system_nvt1, integrator_nvt1, platform)
context_nvt1.setPositions(positions_nve)   # same starting positions as NVE
context_nvt1.setVelocitiesToTemperature(T_K, seed)
displacer_nvt1.displaceToEquilibrium(context_nvt1, lambda_coupling)

# ── Run and track kinetic temperature + dipole ────────────────────────────
dipoles_nvt1 = []
T_kin_nvt1   = []
T_ph_nvt1    = []

kB_kJ = Units.KB_KJMOL_PER_K   # kJ/mol/K
m_ph = system_nvt1.getParticleMass(2).value_in_unit(unit.dalton)

for step in range(N_steps):
    integrator_nvt1.step(1)
    state = context_nvt1.getState(getPositions=True, getVelocities=True)
    pos = state.getPositions(asNumpy=True)
    vel = state.getVelocities(asNumpy=True).value_in_unit(unit.nanometer / unit.picosecond)
    d = charges_single[0] * pos[0] + charges_single[1] * pos[1]
    dipoles_nvt1.append(d)
    # Molecular kinetic temperature using Bussi DOF = 3*N_mol - 3 = 3
    m0 = system_nvt1.getParticleMass(0).value_in_unit(unit.dalton)
    m1 = system_nvt1.getParticleMass(1).value_in_unit(unit.dalton)
    KE_mol = 0.5 * m0 * np.sum(vel[0]**2) + 0.5 * m1 * np.sum(vel[1]**2)
    T_kin = 2.0 * KE_mol / (3 * kB_kJ)  # K
    T_kin_nvt1.append(T_kin)
    # Photon in-plane T (2 DOF, cavity plane y-z); not Bussi-thermostatted
    KE_ph = 0.5 * m_ph * (vel[2, 1]**2 + vel[2, 2]**2)
    T_ph_nvt1.append(2.0 * KE_ph / (2 * kB_kJ))

dipoles_nvt1 = np.array(dipoles_nvt1)
T_kin_nvt1   = np.array(T_kin_nvt1)
T_ph_nvt1    = np.array(T_ph_nvt1)

fig, axes = plt.subplots(3, 1, figsize=(10, 7))
axes[0].plot(t_ps, dipoles_nvt1[:, 0], lw=0.6)
axes[0].set_ylabel('dipole (e·nm)')
axes[0].set_title('NVT single A–A dimer — dipole x-component')
axes[1].plot(t_ps, T_kin_nvt1, lw=0.6, color='C1')
axes[1].axhline(T_K, color='k', ls='--', label=f'T_bath = {T_K} K')
axes[1].set_ylabel('T_kin mol (K)')
axes[1].legend()
axes[2].plot(t_ps, T_ph_nvt1, lw=0.6, color='C2')
axes[2].set_xlabel('time (ps)')
axes[2].set_ylabel('T_kin photon (K, 2 DOF)')
axes[2].set_title('Photon is colder than bath (not thermostatted)')
plt.tight_layout()
plt.show()

print(f"Mean molecular T_kin = {T_kin_nvt1.mean():.1f} K  (target {T_K} K)")
print(f"Mean photon T_kin (2 DOF, y-z) = {T_ph_nvt1.mean():.1f} K  (below bath is expected at small λ)")

---
## Section 3 — NVT two dimers (A–A + B–B, LJ + Coulomb, bond-excluded)

Now we add **inter-molecular interactions**:  
- `NonbondedForce` — Coulomb (point charges ±0.3e on each atom, `NoCutoff` because the system is in vacuum)
- `CustomNonbondedForce` — Kob–Andersen Lennard-Jones with a 3×3 look-up table (types A, B, photon)

**Bond exclusions** prevent the two atoms of the same dimer from interacting via LJ or Coulomb  
(their interaction is already captured by the harmonic bond force).

### The 3×3 ε/σ table
| | A (type 0) | B (type 1) | photon (type 2) |
|--|--|--|--|
| **A** | ε_AA, σ_AA | ε_AB, σ_AB | 0, 1 |
| **B** | ε_AB, σ_AB | ε_BB, σ_BB | 0, 1 |
| **photon** | 0, 1 | 0, 1 | 0, 1 |

The photon row/column is all-zero so the photon has no LJ interaction with the molecules.

In [ ]:
# ── Build a two-dimer + photon System ────────────────────────────────────
# Particle indices:
#   0, 1 — A-A dimer  (type 0)
#   2, 3 — B-B dimer  (type 1)
#   4    — photon     (type 2)

system_2d = openmm.System()
system_2d.addParticle(MASS_A)           # 0
system_2d.addParticle(MASS_A)           # 1
system_2d.addParticle(MASS_B)           # 2
system_2d.addParticle(MASS_B)           # 3
system_2d.addParticle(PHOTON_MASS_AMU)  # 4

# ── Harmonic bonds ───────────────────────────────────────────────────────
k_bb_omm  = K_BB_AU  * H2K / B2NM**2
r0_bb_omm = R0_BB_AU * B2NM

bond_force_2d = openmm.HarmonicBondForce()
bond_force_2d.addBond(0, 1, r0_aa_omm, k_aa_omm)  # A-A
bond_force_2d.addBond(2, 3, r0_bb_omm, k_bb_omm)  # B-B
system_2d.addForce(bond_force_2d)

# ── Coulomb: NoCutoff (vacuum), charges ±0.3e ─────────────────────────────
# addParticle(charge, sigma_nm, epsilon_kJmol)  — sigma/epsilon are dummy (no LJ here)
coulomb_2d = openmm.NonbondedForce()
coulomb_2d.setNonbondedMethod(openmm.NonbondedForce.NoCutoff)
coulomb_2d.addParticle(-CHARGE_MAG, 0.1, 0.0)   # A−
coulomb_2d.addParticle(+CHARGE_MAG, 0.1, 0.0)   # A+
coulomb_2d.addParticle(-CHARGE_MAG, 0.1, 0.0)   # B−
coulomb_2d.addParticle(+CHARGE_MAG, 0.1, 0.0)   # B+
coulomb_2d.addParticle( 0.0,        0.1, 0.0)   # photon (no charge)
# Bond exclusions — zero out intra-dimer Coulomb
coulomb_2d.addException(0, 1, 0.0, 1.0, 0.0)   # A-A bond
coulomb_2d.addException(2, 3, 0.0, 1.0, 0.0)   # B-B bond
system_2d.addForce(coulomb_2d)

# ── KA Lennard-Jones: 3×3 table (A=0, B=1, photon=2) ─────────────────────
eps_aa, sig_aa = EPS_AA_AU * H2K,  SIG_AA_AU * B2NM
eps_ab, sig_ab = EPS_AB_AU * H2K,  SIG_AB_AU * B2NM
eps_bb, sig_bb = EPS_BB_AU * H2K,  SIG_BB_AU * B2NM

lj_2d = openmm.CustomNonbondedForce(
    "lj - ljcut;"
    "lj = 4*eps*((sig/r)^12 - (sig/r)^6);"
    "ljcut = 4*eps*((sig/rc)^12 - (sig/rc)^6);"
    "eps = epsfun(type1, type2);"
    "sig = sigfun(type1, type2)"
)
lj_2d.addPerParticleParameter("type")
# Use a large cutoff so all pairs are included (vacuum system)
lj_2d.setNonbondedMethod(openmm.CustomNonbondedForce.NoCutoff)
lj_2d.addGlobalParameter("rc", 100.0)   # effectively no cutoff

eps_table = [eps_aa, eps_ab, 0.0,
             eps_ab, eps_bb, 0.0,
             0.0,   0.0,   0.0]
sig_table = [sig_aa, sig_ab, 1.0,
             sig_ab, sig_bb, 1.0,
             1.0,   1.0,   1.0]
lj_2d.addTabulatedFunction("epsfun", openmm.Discrete2DFunction(3, 3, eps_table))
lj_2d.addTabulatedFunction("sigfun", openmm.Discrete2DFunction(3, 3, sig_table))

lj_2d.addParticle([0.0])   # A−
lj_2d.addParticle([0.0])   # A+
lj_2d.addParticle([1.0])   # B−
lj_2d.addParticle([1.0])   # B+
lj_2d.addParticle([2.0])   # photon
# Bond exclusions — no intra-dimer LJ
lj_2d.addExclusion(0, 1)   # A-A bond
lj_2d.addExclusion(2, 3)   # B-B bond
system_2d.addForce(lj_2d)

# ── CavityForce + displacer ───────────────────────────────────────────────
cavity_force_2d = openmm.CavityForce(4, omegac_au, lambda_coupling, PHOTON_MASS_AMU)
system_2d.addForce(cavity_force_2d)

displacer_2d = openmm.CavityParticleDisplacer(4, omegac_au, PHOTON_MASS_AMU)
displacer_2d.setSwitchOnStep(2**31 - 1)
system_2d.addForce(displacer_2d)

# ── Bussi thermostat on all molecular atoms ───────────────────────────────
DualThermostat.setup_bussi_for_system(
    system_2d, molecular_indices=[0, 1, 2, 3], temperature_K=T_K,
    tau_ps=1.0, random_number_seed=seed,
)

print("Forces:", [type(system_2d.getForce(i)).__name__ for i in range(system_2d.getNumForces())])

In [ ]:
# ── Initial positions: A-A at x=0, B-B offset by ~15 Bohr along y ─────────
sep_nm = 15.0 * B2NM  # 15 Bohr inter-dimer separation
positions_2d = [
    openmm.Vec3(-half_r0,         0, 0) * unit.nanometer,  # A−
    openmm.Vec3(+half_r0,         0, 0) * unit.nanometer,  # A+
    openmm.Vec3(-r0_bb_omm/2, sep_nm, 0) * unit.nanometer, # B−
    openmm.Vec3(+r0_bb_omm/2, sep_nm, 0) * unit.nanometer, # B+
    openmm.Vec3(           0,      0, 0) * unit.nanometer,  # photon
]

integrator_2d = openmm.VerletIntegrator(dt_ps)
context_2d = openmm.Context(system_2d, integrator_2d, platform)
context_2d.setPositions(positions_2d)
context_2d.setVelocitiesToTemperature(T_K, seed)
displacer_2d.displaceToEquilibrium(context_2d, lambda_coupling)

# ── Run and collect total molecular dipole ────────────────────────────────
charges_2d = np.array([-CHARGE_MAG, +CHARGE_MAG, -CHARGE_MAG, +CHARGE_MAG])

dipoles_2d = []
for _ in range(N_steps):
    integrator_2d.step(1)
    state = context_2d.getState(getPositions=True)
    pos = state.getPositions(asNumpy=True)
    d = np.dot(charges_2d, pos[:4])   # sum over molecular atoms only
    dipoles_2d.append(d)

dipoles_2d = np.array(dipoles_2d)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_ps, dipoles_2d[:, 0], lw=0.5, label='A-A + B-B')
ax.plot(t_ps, dipoles_nvt1[:, 0], lw=0.5, alpha=0.6, label='single A-A')
ax.set_xlabel('time (ps)')
ax.set_ylabel('d_x (e·nm)')
ax.set_title('NVT two dimers vs single dimer — x-dipole')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 4 — NVT N-many dimers (use the library builder)

For larger systems we delegate the repetitive construction to `build_mka_system()`,
which places the correct number of A–A and B–B dimers on a lattice,
sets up PME Coulomb (periodic), and adds the KA LJ table with bond exclusions.
We then attach the cavity manually.

Increase `N_MOL` to see how the spectrum narrows with system size.

In [ ]:
# The mKA builder uses periodic boundaries (PME) with a 15 Bohr cutoff.
# The box must be > 30 Bohr (= 2x cutoff), requiring N_MOL >= 106.
# Raise N_MOL toward 250 for production-scale statistics.
N_MOL = 110   # minimum safe value; 250 for paper-scale statistics

# build_mka_system returns (system, positions, n_mol_particles)
system_N, positions_N, n_atoms = build_mka_system(
    num_molecules=N_MOL,
    seed=seed,
    sample_bonds_at_T=T_K,   # sample initial bond lengths from a thermal distribution
)

# add_cavity_particle appends the photon to the existing system and positions list
# and registers it as a ghost (q=0, ε=0) in every NonbondedForce already present.
cavity_idx = add_cavity_particle(system_N, positions_N)

print(f"System: {n_atoms} molecular atoms + 1 photon  (particle index {cavity_idx})")
print(f"Box edge: {box_au_for_num_molecules(N_MOL) * B2NM * 10:.2f} Å")

In [ ]:
# ── Attach the cavity physics ─────────────────────────────────────────────
cavity_force_N = openmm.CavityForce(cavity_idx, omegac_au, lambda_coupling, PHOTON_MASS_AMU)
system_N.addForce(cavity_force_N)

displacer_N = openmm.CavityParticleDisplacer(cavity_idx, omegac_au, PHOTON_MASS_AMU)
displacer_N.setSwitchOnStep(2**31 - 1)
system_N.addForce(displacer_N)

# ── Bussi thermostat on all molecular atoms ───────────────────────────────
DualThermostat.setup_bussi_for_system(
    system_N,
    molecular_indices=list(range(n_atoms)),
    temperature_K=T_K,
    tau_ps=1.0,
    random_number_seed=seed,
)

# ── Integrator + Context ──────────────────────────────────────────────────
integrator_N = openmm.VerletIntegrator(dt_ps)
platform_N = platform  # CUDA preferred, set in Section 0
context_N = openmm.Context(system_N, integrator_N, platform_N)
context_N.setPositions(positions_N)
context_N.setVelocitiesToTemperature(T_K, seed)
displacer_N.displaceToEquilibrium(context_N, lambda_coupling)

In [ ]:
# ── Equilibration (optional short burn-in) ────────────────────────────────
N_equil = 2_000
integrator_N.step(N_equil)
print(f"Equilibrated {N_equil} steps ({N_equil * dt_ps:.1f} ps)")

# ── Production run — collect molecular dipole d(t) ────────────────────────
# Build the charge array once
charges_N = np.tile([-CHARGE_MAG, +CHARGE_MAG], N_MOL)   # alternating −/+ for each dimer

dipoles_N = []
for _ in range(N_steps):
    integrator_N.step(1)
    state = context_N.getState(getPositions=True)
    pos = state.getPositions(asNumpy=True)   # (n_atoms+1, 3)
    # sum over molecular atoms only (exclude photon at index cavity_idx)
    d = np.dot(charges_N, pos[:n_atoms])     # e·nm
    dipoles_N.append(d)

dipoles_N = np.array(dipoles_N)   # (N_steps, 3)
print(f"Collected {N_steps} dipole samples  ({N_steps * dt_ps:.1f} ps)")

In [ ]:
# Quick sanity check: kinetic temperature over the production run
state_check = context_N.getState(getEnergy=True)
KE = state_check.getKineticEnergy().value_in_unit(unit.kilojoules_per_mole)
# T = 2 KE / (kB * DOF),  DOF = 3*n_atoms - 3 (COM removed)
DOF = 3 * n_atoms - 3
T_check = 2.0 * KE / (DOF * Units.KB_KJMOL_PER_K)
print(f"Final kinetic temperature: {T_check:.1f} K  (target {T_K} K)")

---
## Section 5 — Dipole moment spectrum (manual FFT)

The **infrared absorption spectrum** is proportional to the Fourier transform of the  
dipole autocorrelation function (linear response theory):

$$I(\omega) \propto \omega \int_0^\infty \langle \mathbf{d}(0) \cdot \mathbf{d}(t) \rangle \, e^{i\omega t} \, dt$$

We compute this step by step below.

**Tip**: the cells below work on *any* of the four dipole arrays collected above —
`dipoles_nve`, `dipoles_nvt1`, `dipoles_2d`, or `dipoles_N`.  
Change `dipoles_use` to compare them.

In [ ]:
# Choose which trajectory to analyse
dipoles_use = dipoles_N    # <- change to dipoles_nve, dipoles_nvt1, or dipoles_2d

# Work with the x-component (most informative for a dimer aligned along x)
d_x = dipoles_use[:, 0].copy()
N_sig = len(d_x)

# Step 1 — zero-mean the signal
d_x -= d_x.mean()
print(f"Signal: {N_sig} points,  duration = {N_sig * dt_fs * 1e-3:.2f} ps")

In [ ]:
# Step 2 — autocorrelation via the Wiener–Khinchin theorem
# ACF(τ) = ⟨d(0) d(τ)⟩  computed as the inverse FFT of |FFT(d)|²
fft_d   = np.fft.rfft(d_x, n=2 * N_sig)         # zero-pad to 2N to avoid circular correlation
acf     = np.fft.irfft(np.abs(fft_d)**2)[:N_sig] # take first N points
acf    /= acf[0]                                  # normalise: C(0) = 1

lag_ps = np.arange(N_sig) * dt_ps

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(lag_ps[:N_sig//4], acf[:N_sig//4], lw=0.8)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('lag time (ps)')
ax.set_ylabel('C(τ)')
ax.set_title('Normalised dipole autocorrelation function')
plt.tight_layout()
plt.show()

In [ ]:
# Step 3 — power spectrum  I(ω) ∝ ω · FFT[C(τ)]
spectrum_raw = np.abs(np.fft.rfft(acf))**2

# Frequency axis in cm⁻¹: f(Hz) = k / (N * dt_s),  then cm⁻¹ = f / c_cm
dt_s   = dt_fs * 1e-15                                   # fs → s
freqs  = np.fft.rfftfreq(N_sig, d=dt_s)                  # Hz
freqs_cm1 = freqs / 3e10                                  # cm⁻¹

# Apply the ω prefactor from linear response theory
spectrum = freqs_cm1 * spectrum_raw

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(freqs_cm1, spectrum / spectrum.max(), lw=0.9, label=f'λ = {lambda_coupling}')
ax.axvline(OMEGA_C_CM1, color='r', ls='--', lw=1.2, label=f'ω_c = {OMEGA_C_CM1} cm⁻¹')
ax.set_xlim(0, 4000)
ax.set_xlabel('Frequency (cm⁻¹)')
ax.set_ylabel('Intensity (arb.)')
ax.set_title('Molecular dipole spectrum')
ax.legend()
plt.tight_layout()
plt.show()

# Find the peak
mask = (freqs_cm1 > 500) & (freqs_cm1 < 3000)
peak_idx = np.argmax(spectrum[mask])
peak_cm1 = freqs_cm1[mask][peak_idx]
print(f"Peak frequency: {peak_cm1:.0f} cm⁻¹   (cavity ω_c = {OMEGA_C_CM1:.0f} cm⁻¹)")

### What you should see

- For **λ = 0** (no coupling): the peak sits at the bare A–A stretch frequency, slightly red-shifted from ω_c = 1560 cm⁻¹ by anharmonicity.
- For **λ > 0**: the peak splits into a lower-polariton (LP) and upper-polariton (UP) branch  
  separated by the Rabi splitting ΔΩ ≈ 2λ√N · ω_c.
  Try `lambda_coupling = 0.03` with `N_MOL = 250` to resolve it clearly.

### Going further

The code already contains higher-level helpers you can use in production:

```python
from openmm.cavitymd.trackers import EnergyTracker, ElapsedTimeTracker
from openmm.cavitymd.simulation import CavityMDSimulation
```

`CavityMDSimulation` handles checkpointing, adaptive timestepping, and F(k,t) tracking automatically.
The tutorial above shows you the building blocks that live underneath it.

---
## Reference — physical constants and conversions

| Quantity | Symbol | Value | a.u. |
|----------|--------|-------|---------|
| Cavity frequency | ω_c | 1560 cm⁻¹ | 7.11 × 10⁻³ Ha |
| A–A bond length | r₀_AA | 2.282 Bohr | 0.1208 nm |
| A–A bond stiffness | K_AA | 0.732 Ha/Bohr² | — |
| Charge magnitude | q | ±0.3 e | — |
| A-atom mass | m_A | 16.0 amu | — |
| Photon mass | m_ph | 1/1822.9 amu | 1 a.u. |
| 1 Bohr | — | — | 0.05292 nm |
| 1 Hartree | — | — | 2625.5 kJ/mol |
| 1 a.u. time | — | — | 2.419 × 10⁻⁵ ps |